In [27]:
# importing libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential

# Reading the dataset
train = pd.read_csv("engtamilTrain.csv")
train = train.drop(["Unnamed: 0"], axis =1)   # Removing the unnamed columns in the dataset
eng_sentences = train["en"]                   # Assigning en column for eng_sentences
tamil_sentences = train["ta"]                 # Assigning ta column for tamil_sentences
eng_sentences = eng_sentences.head(1000)      # Taking first 1000 rows for processing
tamil_sentences = tamil_sentences.head(1000)

In [28]:
# Function to add SOS and EOS to the statement
def addSosEos(seriesSentence):
    # Define the <SOS> and <EOS> tokens
    sos_token = "<SOS>"
    eos_token = "<EOS>"

    # Add <SOS> and <EOS> tokens to each statement
    statements_with_tokens = {f"{sos_token} {statement} {eos_token}" for statement in seriesSentence}

    english_sent = []     # Declaring an empty list 
    # Print the statement with tokens
    for statement in statements_with_tokens:
        english_sent.append(statement)        # if the statement is in ststement with token then do append it in english_sent
        print(statement)
    return english_sent

In [29]:
# Function call to print the statements with SOS and EOS
english_sent_SE = addSosEos(eng_sentences)    
tamil_sent_SE = addSosEos(tamil_sentences)

<SOS> In these conditions, it is highly significant that the Balmoral Estate Action Committee declared: 'We make a special appeal to our Sinhala class brothers and sisters.
 <EOS>
<SOS> None of the arguments advanced by the Bush administration and its media apologists - quite aside from their underlying lack of credibility - provide a legal justification for war.
 <EOS>
<SOS> However, he would not listen to her voice: but, being stronger than she, forced her, and lay with her.
 <EOS>
<SOS> While the US-led Coalition Provisional Authority (CPA) is seeking to put the best possible light on the deal, Iraqis are widely interpreting it as a tactical retreat by the US in the face of the uprising in the Shiite south that has raged since early April, and a victory for Sadr.
 <EOS>
<SOS> Not surprisingly it was the Sinhala extremists - the JVP and Sihala Urumaya (SU) - who made the largest gains.
 <EOS>
<SOS> Following its success, he acted as villain in many films.
 <EOS>
<SOS> Nor did her pre

In [30]:
# importing the libraries
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [31]:
# Tokenize the English and Tamil sentences
english_tokenizer = Tokenizer(filters="")
english_tokenizer.fit_on_texts(english_sent_SE)              # Convert text to words using tokenizer
english_vocab_size = len(english_tokenizer.word_index) + 1   # Adding +1 to find vocab size
english_sequences = english_tokenizer.texts_to_sequences(english_sent_SE) # Making word index dictonary

In [32]:
tamil_tokenizer = Tokenizer(filters="")
print(tamil_tokenizer)
tamil_tokenizer.fit_on_texts(tamil_sent_SE)
tamil_vocab_size = len(tamil_tokenizer.word_index) + 1
print(tamil_vocab_size)
tamil_sequences = tamil_tokenizer.texts_to_sequences(tamil_sent_SE)
print(tamil_sequences)

9922
[[1, 64, 884, 1836, 885, 1837, 1838, 1839, 5, 1840, 1841, 886, 1842, 1843, 887, 1844, 111, 1845, 1846, 888, 551, 1847, 889, 2], [1, 1848, 2], [1, 1849, 1850, 1851, 552, 2], [1, 26, 27, 1852, 58, 1853, 188, 1854, 1855, 148, 382, 233, 2], [1, 1856, 189, 1857, 553, 1858, 1859, 1860, 1861, 1862, 1863, 2], [1, 1864, 1865, 1866, 890, 69, 1867, 1868, 1869, 891, 1870, 2], [1, 101, 149, 892, 383, 893, 17, 1871, 1872, 1873, 1874, 1875, 1876, 91, 1877, 1878, 5, 1879, 190, 1880, 150, 65, 894, 1881, 1882, 2], [1, 282, 554, 555, 1883, 5, 1884, 1885, 1886, 5, 78, 384, 1887, 1888, 1889, 2], [1, 17, 1890, 1891, 283, 11, 191, 1892, 1893, 1894, 79, 1895, 1896, 1897, 385, 1898, 92, 2], [1, 35, 64, 1899, 1900, 2], [1, 1901, 192, 1902, 895, 896, 44, 1903, 1904, 1905, 112, 1906, 1907, 2], [1, 193, 556, 386, 4, 387, 1908, 557, 127, 1909, 1910, 897, 23, 898, 1911, 1912, 1913, 899, 1914, 10, 1915, 12, 1916, 1917, 2], [1, 11, 3, 1918, 1919, 128, 1920, 1921, 900, 558, 388, 1922, 1923, 2], [1, 1924, 26, 284, 

In [33]:
# Intializing Input and output seq length
max_input_seq_length=20
max_output_seq_length=20

In [34]:
# Pad Sentences to a fixed length
input_sequences = pad_sequences(english_sequences, maxlen = max_input_seq_length, padding = "post")
output_sequences = pad_sequences(tamil_sequences, maxlen = max_output_seq_length, padding = "post")

In [35]:
# preparing the decoder input and output for teacher forcing

decoder_input_sequences = np.zeros_like(output_sequences) # Creating array of zeros as output sequences
print(decoder_input_sequences)
decoder_input_sequences[:, 1] = output_sequences[:, -1]   # Filling the second columns with last values
print(decoder_input_sequences[:,1])
decoder_input_sequences[:, 0] = tamil_tokenizer.word_index['<sos>']   # Convert sos token at start
print(decoder_input_sequences[:,0])
decoder_output_sequences = np.eye(tamil_vocab_size)[output_sequences] # Identify the matrix and convert to one hot encoding
print(decoder_output_sequences)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
[2 0 0 0 0 0 2 0 0 0 0 2 0 2 0 0 0 0 2 2 2 0 2 2 0 2 0 0 2 0 0 0 0 0 0 0 2
 2 0 0 2 2 0 2 0 0 0 2 0 2 2 0 0 0 0 2 0 0 2 2 0 0 0 0 0 0 2 2 0 0 0 2 2 2
 0 0 2 0 0 2 0 2 0 0 0 2 2 2 0 2 0 2 0 0 0 2 0 0 2 0 0 2 0 0 0 2 2 0 0 0 2
 2 0 2 0 0 0 0 0 0 2 2 0 2 0 2 2 2 0 2 0 0 0 2 2 0 2 0 0 0 0 0 0 0 2 0 0 0
 2 0 2 2 0 0 0 2 0 2 0 0 0 2 2 0 2 0 0 0 0 0 2 0 0 2 0 2 0 0 0 0 2 2 0 0 0
 2 0 0 0 0 2 0 0 0 2 2 0 0 2 0 2 2 0 0 2 0 0 0 0 2 0 2 0 0 0 2 0 2 0 0 0 2
 0 0 2 2 0 2 0 2 2 0 0 0 2 0 0 2 0 0 2 2 2 2 0 0 2 0 0 2 0 2 0 0 2 2 0 2 2
 0 0 2 0 0 0 2 2 2 0 2 0 2 0 0 0 0 2 2 0 0 2 0 0 2 2 0 0 0 2 0 0 0 0 2 2 0
 2 0 2 2 2 0 0 0 2 0 0 0 0 0 0 0 0 0 2 0 0 2 0 0 0 0 0 0 2 2 0 2 2 2 0 2 2
 0 0 0 0 0 0 2 0 0 0 0 2 2 0 2 0 2 2 0 0 0 0 2 0 0 0 0 0 0 2 0 0 0 0 2 2 2
 0 2 2 0 2 0 0 0 2 0 0 0 2 0 2 0 2 2 0 2 0 2 0 0 2 0 0 0 2 0 0 2 0 0 0 0 0
 0 2 0 0 0 0 2 2 2 0 0 0 0 0 0 0 2 2 0 0 0 0 2 2 0 0 0 

In [36]:
# importing gensim to convert vector as numbers
from gensim.models import Word2Vec

# Loading the pre trained model
eng_model = Word2Vec.load('engmodel.bin') 
tam_model = Word2Vec.load('tammodel.bin')

In [37]:
# Create an embedding matrix
def create_embedding_matrix(word2vec_model, tokenizer, vocab_size):
    embedding_matrix = np.zeros((vocab_size, word2vec_model.vector_size)) # create empty embedding matrix
    for word, i in tokenizer.word_index.items():                          # Looping all the words
        try:
            embedding_vector = word2vec_model.wv[word]                    # Converting the word using word2vector and saving it in embedding vector
            embedding_matrix[i] = embedding_vector
        except KeyError:
            pass # Words not found in the embedding index will be all zeros
    return embedding_matrix

# FUnction call
eng_embedding_matrix = create_embedding_matrix(eng_model, english_tokenizer, english_vocab_size)
print(eng_embedding_matrix)
tam_embedding_matrix = create_embedding_matrix(tam_model, tamil_tokenizer, tamil_vocab_size)

[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [-0.26573214  0.61581212  0.10475288 ... -0.59400553 -0.18814734
   0.18171166]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [-0.00995158  0.00973144  0.00571215 ... -0.01232108 -0.01111607
  -0.00636867]
 [ 0.00414906  0.01483785 -0.00310082 ... -0.01504056 -0.00335798
   0.01197518]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]


In [38]:
eng_embedding_matrix.shape

(6979, 100)

In [39]:
tam_embedding_matrix.shape

(9922, 100)

In [40]:
# Creating sequence model using various parameters
def create_seq2seq_model(input_vocab_size, output_vocab_size, input_seq_length, output_seq_length, hidden_units, eng_embedding_matrix, tamil_embedding_matrix):
    # Encoder
    from tensorflow.keras.layers import Input
    from tensorflow.keras.models import Model

    # Read the input sentence and converting input word indices to vectors and feeding them for LSTM 
    encoder_inputs = Input(shape=(input_seq_length,))
    encoder_embedding = Embedding(input_vocab_size, hidden_units, weights = [eng_embedding_matrix], trainable = False)(encoder_inputs)
    encoder_lstm, encoder_state_h, encoder_state_c = LSTM(hidden_units, return_state= True)(encoder_embedding)

    # Decoder
    # Read the output sentences and converting output word indices to vectors and feeding them for LSTM 
    decoder_inputs = Input(shape=(output_seq_length,))
    decoder_embedding = Embedding(output_vocab_size, hidden_units, weights = [tam_embedding_matrix], trainable = False)(decoder_inputs)
    decoder_lstm = LSTM(hidden_units, return_sequences=True, return_state= True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state = [encoder_state_h, encoder_state_c])

    # decoder output undergoes neural network activation 
    decoder_dense = Dense(output_vocab_size, activation = 'softmax')
    decoder_outputs = decoder_dense(decoder_outputs)

    # Finally saving the inputs and outputs to model 
    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    return model

In [41]:
# Convert target sequences to one-hot encoded format
target_sequences = tf.keras.utils.to_categorical(output_sequences, num_classes=tamil_vocab_size)

In [42]:
# Creating sequential model
model = create_seq2seq_model(english_vocab_size, tamil_vocab_size, max_input_seq_length, max_output_seq_length, 100, eng_embedding_matrix, tam_embedding_matrix)

In [43]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [44]:
# Fit the model to the data
batch_size = 32
epochs = 10
model.fit([input_sequences, output_sequences],decoder_output_sequences, batch_size=batch_size, epochs=epochs, validation_split=0.2)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 15s 341ms/step - accuracy: 0.2633 - loss: 8.9303 - val_accuracy: 0.2512 - val_loss: 8.3352
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 258ms/step - accuracy: 0.2742 - loss: 7.3520 - val_accuracy: 0.2485 - val_loss: 7.5595
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 397ms/step - accuracy: 0.2727 - loss: 6.5015 - val_accuracy: 0.2505 - val_loss: 7.6796
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 10s 392ms/step - accuracy: 0.2737 - loss: 6.3749 - val_accuracy: 0.2500 - val_loss: 7.7945
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 264ms/step - accuracy: 0.2735 - loss: 6.3090 - val_accuracy: 0.2500 - val_loss: 7.8576
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 249ms/step - accuracy: 0.2732 - loss: 6.2621 - val_accuracy: 0.2505 - val_loss: 7.9354
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 249ms/step - accuracy: 0.2735 - loss: 6.2223 - val_accuracy: 0.2495 - val_loss: 7.9754
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 241ms/step - accuracy: 0.2732 - loss: 6.1859 - val_accuracy:

In [50]:
# Preprocessing the input
input_sentences = "<sos>My aunt Ehammal was also killed at the same time<eos>"

# convert the input sentence to sequence
input_sequence = english_tokenizer.texts_to_sequences([input_sentences])

# Pad the statement to the maximum input sequence length
input_sequence = pad_sequences(input_sequence, maxlen=max_input_seq_length, padding='post')

# Generate predictions
predictions = model.predict([input_sequence, np.zeros((1, max_output_seq_length))])


# Convert predictions to tokens
predicted_tokens = np.argmax(predictions, axis=-1)[0]

# Create index to word mapping fot Tamil Vocabulary
tamil_index_word = {i: w for w,i in tamil_tokenizer.word_index.items()}

# Convert tokens to text
decoded_sentence = []
for token in predicted_tokens:
    if token == 0: # Assuming 0 is the padding token
        continue
    word = tamil_index_word.get(token)
    if word == '<eos>':
        break
    if word is not None:
        decoded_sentence.append(word) 
    else:
        decoded_sentence.append('<unk>')

# Join the words to form the decoded statement
decoded_statement = ' '.join(decoded_sentence)

# print the decoded statement
print(decoded_statement)
        

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
<sos> <sos> <sos> <sos> <sos> மற்றும்


In [51]:
predictions

array([[[1.8362780e-03, 5.0310808e-01, 1.9118005e-03, ...,
         6.3737679e-07, 7.4720060e-07, 6.5884973e-07],
        [9.2130888e-04, 1.0296860e-01, 1.2762632e-03, ...,
         3.1260190e-06, 3.3509921e-06, 3.1560128e-06],
        [9.5286267e-04, 3.3130903e-02, 1.4397135e-03, ...,
         4.3428431e-06, 4.5483139e-06, 4.4534054e-06],
        ...,
        [5.5547756e-01, 6.6048017e-04, 9.9221624e-02, ...,
         4.2064252e-07, 4.2599075e-07, 4.0246775e-07],
        [5.5747408e-01, 6.4943638e-04, 9.9676728e-02, ...,
         4.1613620e-07, 4.2072477e-07, 3.9719663e-07],
        [5.5845028e-01, 6.4310333e-04, 9.9899665e-02, ...,
         4.1373525e-07, 4.1792376e-07, 3.9437955e-07]]],
      shape=(1, 20, 9922), dtype=float32)

In [52]:
predicted_tokens

array([1, 1, 1, 1, 1, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [53]:
input_sequence

array([[5268, 5269,   15,   74,  232,   25,    1,  133,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0]],
      dtype=int32)